In [1]:
### misc
import pandas as pd
import numpy as np
import os
from pathlib import Path
import pickle
import time
from itertools import product

#### graphical
import matplotlib.pyplot as plt
import corner

#### ML
import sklearn
from sklearn.decomposition import PCA
import tensorflow as tf
import keras
from keras import layers

from WMSE import WMSE, WMSE_metric

##### poke gpu
os.environ["CUDA_VISIBLE_DEVICES"]="0"

physical_devices = tf.config.list_physical_devices("GPU") 

tf.config.experimental.set_memory_growth(physical_devices[0], True)

gpu0usage = tf.config.experimental.get_memory_info("GPU:0")["current"]

print("Current GPU usage:\n"
     + " - GPU0: " + str(gpu0usage) + "B\n")

modelpath = '/home/hatte/M4/models'

2025-04-07 14:49:19.618060: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1744033759.629998  210555 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1744033759.633770  210555 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-04-07 14:49:19.647055: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Current GPU usage:
 - GPU0: 0B



I0000 00:00:1744033761.062458  210555 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 17511 MB memory:  -> device: 0, name: NVIDIA RTX A4500, pci bus id: 0000:41:00.0, compute capability: 8.6


In [2]:
def scheduler(epoch, lr,):
    ## Learning rate scheduler
    # Decreases learning rate in-training for stability
    if lr < 1e-5:
        return float(lr)
    else:
        return float(lr * tf.math.exp(-0.000006))

In [3]:
df_full = pd.read_hdf('../grids/Chiara.hdf5', key='df') ## edit for your grid!!

df_full['logLPhot'] = np.log10(df_full['LPhot'])

df_full['lognumax'] = np.log10(df_full['numax']*3090)

df_full['logdnuSer'] = np.log10(df_full['dnuSer']*135)

#### define inputs
inputs = ['massini', 'zini', 'yini', 'alphaMLT', 'age', 'eta', 'alphaFe']

#### define outputs
classical_outputs = ['FeH', 'logLPhot', 'Teff']
astero_outputs = ['numax', 'dnuSer'] 

outputs = classical_outputs+astero_outputs

df = df_full[inputs+outputs]

df_norm = (df - df.min())/(df.max() - df.min())

## check df_norm.describe looks reasonable (min=0, max=1):
df_norm.describe()

#### train/test split with seed 
seed = 42

df_train = df_norm.sample(frac=0.95, random_state=seed)
df_test = df_norm.drop(df_train.index)

df_train_inputs, df_val_inputs, df_train_outputs, df_val_outputs = sklearn.model_selection.train_test_split(df_train[inputs],df_train[outputs], test_size = 0.05, random_state=seed)

print("Training set: ", len(df_train_inputs))
print("Validation set: ", len(df_val_inputs))
print("Test set: ", len(df_test))

Training set:  6754081
Validation set:  355478
Test set:  374187


In [4]:
#unnormed_weights_dict = {'FeH':0.01, 'logLPhot':0.001, 'Teff':10, 'numax':0.001/3090, 'dnuSer':0.0001/135}

unnormed_weights_dict = {'FeH':0.01, 'logLPhot':0.001, 'Teff':1, 'numax':0.0001/3090, 'dnuSer':0.0001/135}

unnormed_weights = list(unnormed_weights_dict.values())

weights = [2*unnormed_weights_dict[i]/(df[i].max() - df[i].min()) for i in outputs]

In [6]:
n_dense_layers = 6

dense_layer_units = 128

Nepochs = 10000

learning_rate = 0.001

model_name = 'smart-logLPhot-numax-dnuSer-exponent-6e-6-Adam'

loss_func = 'WMSE'

df_train_inputs.join(df_train_outputs).to_hdf(f'{modelpath}/long-runs/training-data/training-{model_name}-nlayers-{n_dense_layers}-nunits-{dense_layer_units}-epochs-{Nepochs}-lrate-{learning_rate}-lossfunc-{loss_func}.hdf5', key = 'df')

In [15]:
checkpoint_dir = f'{modelpath}/long-runs/checkpoint/chk-{model_name}-nlayers-{n_dense_layers}-nunits-{dense_layer_units}-epochs-{Nepochs}-lrate-{learning_rate}-lossfunc-{loss_func}.model.keras'

full_model_dir = f'{modelpath}/long-runs/full-model/mod-{model_name}-nlayers-{n_dense_layers}-nunits-{dense_layer_units}-epochs-{Nepochs}-lrate-{learning_rate}-lossfunc-{loss_func}/'

if not os.path.exists(full_model_dir):
    os.makedirs(full_model_dir)

historyfile = f'{modelpath}/long-runs/history/hist-{model_name}-nlayers-{n_dense_layers}-nunits-{dense_layer_units}-epochs-{Nepochs}-lrate-{learning_rate}-lossfunc-{loss_func}.json'
    
cp_callback = tf.keras.callbacks.ModelCheckpoint(filepath = checkpoint_dir, verbose = 1, save_best_only = True, save_freq = 'epoch')

lr_callback = tf.keras.callbacks.LearningRateScheduler(scheduler, )

In [ ]:
######## map out model architecture
#### input layer
nn_input = keras.Input(shape=(len(inputs),))

#### dense layer(s)
for n_dense_layer in range(n_dense_layers):
    if n_dense_layer == 0:
        dense_layer = layers.Dense(dense_layer_units, activation='relu')(nn_input)
    else:
        dense_layer = layers.Dense(dense_layer_units, activation='relu')(dense_layer)

#### output layer
nn_output =  layers.Dense(len(outputs), activation='linear')(dense_layer)

######## store architecture as keras model
model = keras.Model(inputs=nn_input, outputs=nn_output, name=model_name)

tb_callback = tf.keras.callbacks.TensorBoard(log_dir = f'{modelpath}/logs/long-runs/log-{model_name}-nlayers-{n_dense_layers}-nunits-{dense_layer_units}-epochs-{Nepochs}-lrate-{learning_rate}-lossfunc-{loss_func}')

model.compile(loss=WMSE(weights), optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate))

history = model.fit(df_train_inputs,
          df_train_outputs,
          validation_data=(df_val_inputs,df_val_outputs),
          batch_size=2**14, #change higher
          verbose=1,
          epochs=Nepochs,
          shuffle=True, callbacks = [tb_callback, cp_callback, lr_callback]) 

tf.saved_model.save(model, full_model_dir)
hist_df = pd.DataFrame(history.history)
    
with open(os.path.join(modelpath, historyfile), mode="w") as f:
    hist_df.to_json(f)

Epoch 1/10000


I0000 00:00:1744033798.438473  211587 service.cc:148] XLA service 0x79d1e000f560 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1744033798.438499  211587 service.cc:156]   StreamExecutor device (0): NVIDIA RTX A4500, Compute Capability 8.6
2025-04-07 14:49:58.472381: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1744033798.595549  211587 cuda_dnn.cc:529] Loaded cuDNN version 90300
2025-04-07 14:49:58.661020: W external/local_xla/xla/service/gpu/nvptx_compiler.cc:930] The NVIDIA driver's CUDA version is 12.2 which is older than the PTX compiler version 12.5.82. Because the driver is older than the PTX compiler version, XLA is disabling parallel compilation, which may slow down compilation. You should update your NVIDIA driver or use the NVIDIA-provided CUDA forward compatibility packages.
2025-04-07 14:49:59.27493

 67/413 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 53916491776.0000

I0000 00:00:1744033800.556163  211587 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


393/413 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 51967365120.0000

2025-04-07 14:50:02.284611: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_258', 32 bytes spill stores, 32 bytes spill loads

2025-04-07 14:50:02.348332: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_451', 136 bytes spill stores, 136 bytes spill loads

2025-04-07 14:50:02.350944: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_461', 24 bytes spill stores, 48 bytes spill loads

2025-04-07 14:50:02.566451: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_451', 24 bytes spill stores, 24 bytes spill loads

2025-04-07 14:50:02.653016: I external/local_xla/xla/stream_ex

413/413 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 51932905472.0000

2025-04-07 14:50:04.748841: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_30', 32 bytes spill stores, 32 bytes spill loads

2025-04-07 14:50:04.822286: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_30_0', 168 bytes spill stores, 168 bytes spill loads




Epoch 1: val_loss improved from inf to 57334255616.00000, saving model to /home/hatte/M4/models/long-runs/checkpoint/chk-smart-logLPhot-numax-dnuSer-exponent-6e-6-Adam-nlayers-6-nunits-128-epochs-10000-lrate-0.001-lossfunc-WMSE.model.keras
413/413 ━━━━━━━━━━━━━━━━━━━━ 8s 11ms/step - loss: 51931336704.0000 - val_loss: 57334255616.0000 - learning_rate: 9.9999e-04
Epoch 2/10000
401/413 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 50960060416.0000
Epoch 2: val_loss improved from 57334255616.00000 to 50756317184.00000, saving model to /home/hatte/M4/models/long-runs/checkpoint/chk-smart-logLPhot-numax-dnuSer-exponent-6e-6-Adam-nlayers-6-nunits-128-epochs-10000-lrate-0.001-lossfunc-WMSE.model.keras
413/413 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 50949599232.0000 - val_loss: 50756317184.0000 - learning_rate: 9.9999e-04
Epoch 3/10000
398/413 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 50899247104.0000
Epoch 3: val_loss improved from 50756317184.00000 to 49489596416.00000, saving model to /home/hatt

In [16]:
custom_objects =  {'WMSE':WMSE_metric}
model= tf.keras.models.load_model(checkpoint_dir, custom_objects = custom_objects)

In [17]:
model_name_SGD = model_name + '-SGD'

checkpoint_dir = f'{modelpath}/long-runs/checkpoint/chk-{model_name_SGD}-nlayers-{n_dense_layers}-nunits-{dense_layer_units}-epochs-{Nepochs}-lrate-{learning_rate}-lossfunc-{loss_func}.model.keras'

full_model_dir = f'{modelpath}/long-runs/full-model/mod-{model_name_SGD}-nlayers-{n_dense_layers}-nunits-{dense_layer_units}-epochs-{Nepochs}-lrate-{learning_rate}-lossfunc-{loss_func}/'

if not os.path.exists(full_model_dir):
    os.makedirs(full_model_dir)

historyfile = f'{modelpath}/long-runs/history/hist-{model_name_SGD}-nlayers-{n_dense_layers}-nunits-{dense_layer_units}-epochs-{Nepochs}-lrate-{learning_rate}-lossfunc-{loss_func}.json'
    
cp_callback = tf.keras.callbacks.ModelCheckpoint(filepath = checkpoint_dir, verbose = 1, save_best_only = True, save_freq = 'epoch')


In [18]:
unnormed_weights_dict = {'FeH':0.01, 'logLPhot':0.001, 'Teff':1, 'numax':0.001/3090, 'dnuSer':0.001/135}

unnormed_weights = list(unnormed_weights_dict.values())

weights = [2*unnormed_weights_dict[i]/(df[i].max() - df[i].min()) for i in outputs]

In [19]:
n_learning_rate = 0.001

nepochs = 1000 + Nepochs

tb_callback = tf.keras.callbacks.TensorBoard(log_dir = f'{modelpath}/logs/long-runs/log-{model_name_SGD}-nlayers-{n_dense_layers}-nunits-{dense_layer_units}-epochs-{Nepochs}-lrate-{learning_rate}-lossfunc-{loss_func}')

model.compile(loss='MSE', optimizer=tf.keras.optimizers.SGD(learning_rate=n_learning_rate))

In [ ]:
history = model.fit(df_train_inputs,
          df_train_outputs,
          validation_data=(df_val_inputs,df_val_outputs),
          batch_size=2**14, #change higher
          verbose=1,
          epochs=nepochs,
          shuffle=True, callbacks = [tb_callback, cp_callback],
          initial_epoch=Nepochs)



Epoch 10001/11000


2025-04-08 10:36:46.623767: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_319', 112 bytes spill stores, 108 bytes spill loads

2025-04-08 10:36:46.917578: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_319', 3400 bytes spill stores, 3404 bytes spill loads

2025-04-08 10:36:46.966014: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_319', 4716 bytes spill stores, 4704 bytes spill loads



411/413 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0087

2025-04-08 10:36:49.408963: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_319', 76 bytes spill stores, 152 bytes spill loads

2025-04-08 10:36:49.509765: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_319', 572 bytes spill stores, 576 bytes spill loads

2025-04-08 10:36:49.580533: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_319', 584 bytes spill stores, 584 bytes spill loads

2025-04-08 10:36:49.703220: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_319', 4236 bytes spill stores, 4252 bytes spill loads



413/413 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0087
Epoch 10001: val_loss improved from inf to 0.00698, saving model to /home/hatte/M4/models/long-runs/checkpoint/chk-smart-logLPhot-numax-dnuSer-exponent-6e-6-Adam-SGD-nlayers-6-nunits-128-epochs-10000-lrate-0.001-lossfunc-WMSE.model.keras
413/413 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - loss: 0.0087 - val_loss: 0.0070
Epoch 10002/11000
412/413 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0067
Epoch 10002: val_loss improved from 0.00698 to 0.00632, saving model to /home/hatte/M4/models/long-runs/checkpoint/chk-smart-logLPhot-numax-dnuSer-exponent-6e-6-Adam-SGD-nlayers-6-nunits-128-epochs-10000-lrate-0.001-lossfunc-WMSE.model.keras
413/413 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.0067 - val_loss: 0.0063
Epoch 10003/11000
407/413 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0062
Epoch 10003: val_loss improved from 0.00632 to 0.00594, saving model to /home/hatte/M4/models/long-runs/checkpoint/chk-smart-logLPhot-numax-dnuSer-exponent-6e-6-Adam-

In [19]:
tf.saved_model.save(model, full_model_dir)
hist_df = pd.DataFrame(history.history)
    
with open(os.path.join(modelpath, historyfile), mode="w") as f:
    hist_df.to_json(f)

TypeError: this __dict__ descriptor does not support '_DictWrapper' objects

In [30]:
unnormed_weights_dict = {'FeH':0.001, 'logLPhot':0.001, 'Teff':0.1, 'numax':0.001/3090, 'dnuSer':0.001/135}

unnormed_weights = list(unnormed_weights_dict.values())

weights = [2*unnormed_weights_dict[i]/(df[i].max() - df[i].min()) for i in outputs]

In [31]:
n_learning_rate = model.optimizer.get_config()['learning_rate']

nnnepochs = 101000 + 50000

tb_callback = tf.keras.callbacks.TensorBoard(log_dir = f'{modelpath}/logs/long-runs/log-{model_name}-nlayers-{n_dense_layers}-nunits-{dense_layer_units}-epochs-{Nepochs}-lrate-{learning_rate}-lossfunc-{loss_func}')

model.compile(loss=WMSE(weights), optimizer=tf.keras.optimizers.Adam(learning_rate=n_learning_rate))

In [ ]:
history = model.fit(df_train_inputs,
          df_train_outputs,
          validation_data=(df_val_inputs,df_val_outputs),
          batch_size=2**14, #change higher
          verbose=1,
          epochs=nnnepochs,
          shuffle=True, callbacks = [tb_callback, cp_callback, lr_callback],
          initial_epoch=101000)

Epoch 101001/151000
413/413 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 1944704.0000
Epoch 101001: val_loss did not improve from 224604.18750
413/413 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - loss: 1941518.3750 - val_loss: 224674.1875 - learning_rate: 5.4440e-04
Epoch 101002/151000
398/413 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 221558.0000
Epoch 101002: val_loss did not improve from 224604.18750
413/413 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 221667.3281 - val_loss: 231856.9531 - learning_rate: 5.4440e-04
Epoch 101003/151000
405/413 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 228705.0156
Epoch 101003: val_loss did not improve from 224604.18750
413/413 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 228784.2344 - val_loss: 237591.3281 - learning_rate: 5.4440e-04
Epoch 101004/151000
412/413 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 246389.2812
Epoch 101004: val_loss did not improve from 224604.18750
413/413 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 246454.7656 - val_loss: 308530.3750 - learning_rate: 5.443

In [10]:
custom_objects =  {'WMSE':WMSE_metric}
tb_callback = tf.keras.callbacks.TensorBoard(log_dir = f'{modelpath}/logs/long-runs/log-{model_name}-nlayers-{n_dense_layers}-nunits-{dense_layer_units}-epochs-{Nepochs}-lrate-{learning_rate}-lossfunc-{loss_func}')

model = tf.keras.models.load_model(checkpoint_dir, custom_objects = custom_objects)
n_learning_rate = model.optimizer.get_config()['learning_rate']

In [14]:
model.compile(loss = WMSE(weights), optimizer=tf.keras.optimizers.Adam(learning_rate=0.001))

history = model.fit(df_train_inputs,
          df_train_outputs,
          validation_data=(df_val_inputs,df_val_outputs),
          batch_size=2**14, #change higher
          verbose=1,
          epochs=103624+1000,
          shuffle=True, callbacks = [tb_callback, cp_callback, lr_callback],
          initial_epoch=103624)

Epoch 103625/104624


2025-04-07 14:43:23.938591: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_451_0', 32 bytes spill stores, 32 bytes spill loads

2025-04-07 14:43:23.985739: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_461', 20 bytes spill stores, 20 bytes spill loads

2025-04-07 14:43:24.037567: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_451', 192 bytes spill stores, 192 bytes spill loads

2025-04-07 14:43:24.165488: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_461', 416 bytes spill stores, 420 bytes spill loads

2025-04-07 14:43:24.180500: I external/local_xla/xla/strea

407/413 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: nan

2025-04-07 14:43:26.684664: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_451', 24 bytes spill stores, 24 bytes spill loads

2025-04-07 14:43:26.729924: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_451', 136 bytes spill stores, 136 bytes spill loads

2025-04-07 14:43:26.770750: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_461', 24 bytes spill stores, 48 bytes spill loads

2025-04-07 14:43:26.835583: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_461', 412 bytes spill stores, 416 bytes spill loads

2025-04-07 14:43:26.894882: I external/local_xla/xla/stream_

413/413 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: nan
Epoch 103625: val_loss did not improve from inf
413/413 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - loss: nan - val_loss: nan - learning_rate: 9.9999e-04
Epoch 103626/104624
411/413 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: nan
Epoch 103626: val_loss did not improve from inf
413/413 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: nan - val_loss: nan - learning_rate: 9.9999e-04
Epoch 103627/104624
255/413 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: nan

KeyboardInterrupt: 